# Trade Engine Demo: Draft Capital Curve + Net KVS Value

Demonstrates the two pieces just built in `src/trade_engine/`:

1. **`draft_capital_curve.py`** — translates a projected keeper round into an expected VORP figure, using real, realized 2024-2025 fresh-pick outcomes. Two real problems had to be solved before this curve could be built at all, both checked directly rather than assumed:
   - **The crosswalk problem**: `draft_history.parquet` identifies players by Sleeper's `player_id`; `vorp_labels.parquet` identifies them by NFL's `gsis_id`. Joining on Sleeper's own cached `gsis_id` field alone only resolved 100/310 (32%) of fresh picks — misses included Bo Nix, Malik Nabers, Travis Etienne, and ~150 other real, well-known players whose Sleeper record simply has a blank `gsis_id`. A two-tier crosswalk (direct `gsis_id`, then name+position match against `nflreadpy`'s own player table, then direct pass-through for team defenses) recovers 282/310 (91%).
   - **Round sparsity**: real usable-with-VORP counts per round range from 9 (rounds 16-17) to 20 (round 14) across 2024-2025. Rounds 16-17 band together (`min_picks_per_round=10`); every other round, including round 18 sitting exactly at the threshold, stands alone.
2. **`net_value.py`** — `net_KVS_delta = predicted_KVS - keeper_cost_VORP`, where `predicted_KVS` comes from the appropriate tuned QB/RB/WR/TE model (the same `0Xx_model_*.ipynb`/`08x_shap_*.ipynb` models used all night) or the explicitly-labeled K/DEF non-ML heuristic (current-season realized VORP, per `roadmap.md`'s Phase 4 scope decision), and `keeper_cost_VORP` comes from `scripts/project_roster_keeper_costs.py`'s already-validated projected keeper round, converted via the curve above. Every result carries `low_confidence_extreme_delta`/`no_delta_history`/`used_heuristic` explicitly — never silently dropped, even on the heuristic path.

Both pieces are demonstrated below against real, current roster data — the same real-data-eyeball-check standard as every model notebook tonight: real numbers, checked for football sense, surprises reported honestly rather than smoothed over.

## Part 1 — Draft Capital Curve

In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.trade_engine.draft_capital_curve import (
    build_draft_capital_curve,
    build_full_draft_capital_curve,
    resolve_fresh_picks,
    attach_realized_vorp,
)

pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

draft_curve = build_full_draft_capital_curve(REPO_ROOT)
print("Draft capital curve (2024-2025, min_picks_per_round=10):\n")
print(draft_curve.to_string())


Draft capital curve (2024-2025, min_picks_per_round=10):

         avg_vorp  n_picks band_rounds  banded
round                                         
1      107.364286       14        (1,)   False
2       70.524000       15        (2,)   False
3       49.861579       19        (3,)   False
4       32.754667       15        (4,)   False
5        2.320667       15        (5,)   False
6        3.224444       18        (6,)   False
7      -10.859474       19        (7,)   False
8      -18.248571       14        (8,)   False
9       -7.766875       16        (9,)   False
10     -10.253889       18       (10,)   False
11     -12.551111       18       (11,)   False
12     -21.792353       17       (12,)   False
13     -27.937222       18       (13,)   False
14     -13.138500       20       (14,)   False
15     -33.218333       18       (15,)   False
16     -15.471111       18    (16, 17)    True
17     -15.471111       18    (16, 17)    True
18     -14.131000       10       (18,)   False


**Real-data eyeball check**: `avg_vorp` should decline roughly monotonically from round 1 down through the late rounds, since earlier picks are, on average, real fantasy stars and later picks are replacement-level or worse. Rounds 16 and 17 should show identical `avg_vorp`/`n_picks` and a shared `band_rounds = (16, 17)` {EM} the banding decision from the investigation above, visibly working on real data, not just in the unit tests.

### Crosswalk transparency: how many fresh picks actually resolved, and how

In [2]:
import json
import nflreadpy as nfl

draft_history = pd.read_parquet(REPO_ROOT / "data/raw/draft_history.parquet")
with open(REPO_ROOT / "data/raw/players_cache.json") as f:
    sleeper_players = json.load(f)
nfl_players = nfl.load_players().to_pandas()
vorp_labels = pd.read_parquet(REPO_ROOT / "data/processed/vorp_labels.parquet")

resolved = resolve_fresh_picks(draft_history, sleeper_players, nfl_players, seasons=[2024, 2025])
with_vorp = attach_realized_vorp(resolved, vorp_labels)

print("Match method breakdown (how each fresh pick's vorp_key was resolved):")
print(resolved["match_method"].value_counts().to_string())
print(f"\nOf {len(resolved)} fresh picks, {with_vorp['vorp'].notna().sum()} resolved to a real VORP row "
      f"({with_vorp['vorp'].notna().mean():.1%}).")

still_unresolved = resolved[resolved["match_method"] == "unresolved"]["player_id"].unique()
print(f"\n{len(still_unresolved)} player_ids never resolved at all (genuine name collisions / position "
      f"mismatches, not chased further):")
for pid in still_unresolved:
    p = sleeper_players.get(pid, {})
    print(f"  {pid}: {p.get('full_name')} ({p.get('position')})")


Match method breakdown (how each fresh pick's vorp_key was resolved):
match_method
name_fallback      186
direct             100
def_passthrough     19
unresolved           5

Of 310 fresh picks, 282 resolved to a real VORP row (91.0%).

4 player_ids never resolved at all (genuine name collisions / position mismatches, not chased further):
  7670: Joshua Palmer (WR)
  11628: Marvin Harrison (WR)
  12547: Kyle Williams (WR)
  12530: Travis Hunter (DB)


## Part 2 — Net KVS Value, on real current roster players

Six real, currently-rostered players are pulled directly from `data/processed/roster_keeper_table_2027.csv` (— the actual output of `scripts/project_roster_keeper_costs.py`), spanning all four modeled positions plus both K and DEF, and deliberately including at least one player already known from tonight's SHAP work to carry a real confidence caveat (Christian McCaffrey, `low_confidence_extreme_delta` in `06b_model_rb.ipynb`/`08b_shap_rb.ipynb`) and one TE (Kyle Pitts) whose feature set structurally can't represent his real-world context (see `06d_model_te.ipynb`).

In [3]:
roster_keeper_table = pd.read_csv(REPO_ROOT / "data/processed/roster_keeper_table_2027.csv")

demo_players = [
    ("Christian McCaffrey", "RB"),
    ("Ja'Marr Chase", "WR"),
    ("Caleb Williams", "QB"),
    ("Kyle Pitts", "TE"),
    ("Cam Little", "K"),
    ("HOU DEF", "DEF"),  # roster_keeper_table lists this as "Houston Texans" -- vorp_labels
                         # and Sleeper both key DEF rows by team code + " DEF", not the full
                         # team name; a real, minor naming inconsistency worth flagging as-is
                         # rather than silently building a full 32-team name-mapping table for
                         # a 6-player demo.
]

lookup_name = {"HOU DEF": "Houston Texans"}

rows = []
for player_name, position in demo_players:
    roster_name = lookup_name.get(player_name, player_name)
    match = roster_keeper_table[
        (roster_keeper_table["player"] == roster_name) & (roster_keeper_table["position"] == position)
    ]
    if match.empty:
        print(f"WARNING: {roster_name} ({position}) not found in roster_keeper_table_2027.csv -- skipping")
        continue
    rows.append({
        "player_name": player_name,
        "position": position,
        "projected_keeper_round": int(match.iloc[0]["round_lost_if_kept_2027"]),
    })

demo_df = pd.DataFrame(rows)
print("Real players pulled from data/processed/roster_keeper_table_2027.csv:")
print(demo_df.to_string(index=False))


Real players pulled from data/processed/roster_keeper_table_2027.csv:
        player_name position  projected_keeper_round
Christian McCaffrey       RB                       1
      Ja'Marr Chase       WR                       1
     Caleb Williams       QB                       8
         Kyle Pitts       TE                       6
         Cam Little        K                      12
            HOU DEF      DEF                      10


In [4]:
from src.trade_engine.net_value import evaluate_player_trade_value

SEASON = 2025  # each player's real, already-realized 2025 season feeds the 2026 prediction --
                # matching every 06x_model_*.ipynb notebook's own live-prediction convention.

results = []
for _, row in demo_df.iterrows():
    result = evaluate_player_trade_value(
        row["player_name"], SEASON, row["position"], row["projected_keeper_round"],
        draft_curve, REPO_ROOT, vorp_labels=vorp_labels,
    )
    results.append(result)
    print(f"=== {result.player_name} ({result.position}) ===")
    print(f"  predicted_KVS:          {result.predicted_kvs:.2f}")
    print(f"  keeper_cost_VORP:       {result.keeper_cost_vorp:.2f}  (projected round {row['projected_keeper_round']})")
    print(f"  net_KVS_delta:          {result.net_kvs_delta:.2f}")
    print(f"  low_confidence_extreme_delta: {result.low_confidence_extreme_delta}")
    print(f"  no_delta_history:             {result.no_delta_history}")
    print(f"  used_heuristic:               {result.used_heuristic}")
    print(f"  reliability_tier:             {result.reliability_tier}")
    if result.caveats:
        print("  CAVEATS:")
        for c in result.caveats:
            print(f"    - {c}")
    print()


=== Christian McCaffrey (RB) ===
  predicted_KVS:          74.67
  keeper_cost_VORP:       107.36  (projected round 1)
  net_KVS_delta:          -32.70
  low_confidence_extreme_delta: True
  no_delta_history:             False
  used_heuristic:               False
  reliability_tier:             model
  CAVEATS:
    - low_confidence_extreme_delta: |vorp_delta_yoy| > 100 -- this region has confirmed, held-out degraded accuracy at every modeled position (see 08a-08d_shap_*.ipynb); treat the exact predicted_KVS number with real skepticism, not just the direction.

=== Ja'Marr Chase (WR) ===
  predicted_KVS:          76.33
  keeper_cost_VORP:       107.36  (projected round 1)
  net_KVS_delta:          -31.04
  low_confidence_extreme_delta: False
  no_delta_history:             False
  used_heuristic:               False
  reliability_tier:             model



=== Caleb Williams (QB) ===
  predicted_KVS:          -28.61
  keeper_cost_VORP:       -18.25  (projected round 8)
  net_KVS_delta:          -10.36
  low_confidence_extreme_delta: False
  no_delta_history:             False
  used_heuristic:               False
  reliability_tier:             model



=== Kyle Pitts (TE) ===
  predicted_KVS:          19.80
  keeper_cost_VORP:       3.22  (projected round 6)
  net_KVS_delta:          16.58
  low_confidence_extreme_delta: False
  no_delta_history:             False
  used_heuristic:               False
  reliability_tier:             model

=== Cam Little (K) ===
  predicted_KVS:          23.25
  keeper_cost_VORP:       -21.79  (projected round 12)
  net_KVS_delta:          45.04
  low_confidence_extreme_delta: False
  no_delta_history:             False
  used_heuristic:               True
  reliability_tier:             heuristic
  CAVEATS:
    - used_heuristic: predicted_KVS comes from the K/DEF non-ML heuristic (current-season realized VORP), not a tuned model -- per roadmap.md's Phase 4 scope decision, no XGBoost model or SHAP explainer exists for K/DEF. Do not read this with the same confidence as a QB/RB/WR/TE prediction.

=== HOU DEF (DEF) ===
  predicted_KVS:          45.25
  keeper_cost_VORP:       -10.25  (projected round 1

**Real-data eyeball check, player by player:**

- **Christian McCaffrey (RB)**: `net_KVS_delta = -32.70`, and correctly carries `low_confidence_extreme_delta` — his real `vorp_delta_yoy` sits past the 100 threshold, the exact same region `06b_model_rb.ipynb`/`08b_shap_rb.ipynb` already documented this model mishandling for this specific player. A negative net value here should be read with real, specific skepticism, not taken at face value — the flag is doing exactly what it's for.
- **Ja'Marr Chase (WR)**: `net_KVS_delta = -31.04`, no flags. Makes sense on its own terms: round 1's average realized VORP (107.4) is a very high bar — it includes every round-1 bust and every round-1 superstar — and even a strong, unflagged prediction (76.3) falls short of it.
- **Caleb Williams (QB)**: `net_KVS_delta = -10.36`, no flags, but both `predicted_KVS` (-28.6) and `keeper_cost_VORP` (-18.2) are negative. Worth naming plainly rather than smoothing over: this is a real, fairly pessimistic call for a former #1 pick, and this notebook does not independently re-verify his specific feature row the way tonight's other case studies did — it's presented as the real model output, not a football-sense-checked one.
- **Kyle Pitts (TE)**: `net_KVS_delta = +16.58`, no flags. `predicted_KVS` (19.8) matches `08d_shap_te.ipynb`'s own real output for this exact row exactly, which is itself a useful cross-check that this module's live feature-building reproduces that notebook's pipeline correctly. `08d`'s own real-world caveat about Pitts (resolved trade + new contract, which the model can't see) argues the model's prediction may be *understating* him — which would mean this positive net value is, if anything, a conservative floor, not an inflated one.
- **Cam Little (K)** and **HOU DEF**: the two largest `net_KVS_delta` values (+45.0 and +55.5) in the whole table, and both are `used_heuristic=True` — current-season persistence, not a real predictive model. This is worth flagging plainly, not just noting the flag exists: a reader sorting this table by `net_KVS_delta` would see HOU DEF at the top, but `roadmap.md`'s own Phase 4 scope decision is that DEF's year-to-year persistence is barely better than noise (Spearman 0.188-0.307), and K is excluded for the same reason. The two largest numbers in this demo carry the least real evidence behind them — exactly the opposite of what their ranking would suggest at a glance, and exactly why the `used_heuristic` caveat exists. **Fix made directly in response to this finding**: `net_value.py`'s `NetValueResult` now carries a first-class `reliability_tier` field (`"model"` vs. `"heuristic"`), queryable/filterable directly rather than buried in the `caveats` text, and a new `group_by_reliability_tier()` function returns tiers as **separate, independently-sorted lists** rather than one blended numeric sort — so a future ranking UI cannot accidentally reproduce the exact HOU DEF/Cam Little problem just noted by sorting all six players together. Demonstrated directly below, replacing the single blended table.

### Summary table, sorted by net_KVS_delta

In [5]:
from src.trade_engine.net_value import group_by_reliability_tier, RELIABILITY_TIER_MODEL, RELIABILITY_TIER_HEURISTIC

def _to_table(tier_results):
    return pd.DataFrame([
        {
            "player": r.player_name, "position": r.position,
            "predicted_KVS": round(r.predicted_kvs, 1), "keeper_cost_VORP": round(r.keeper_cost_vorp, 1),
            "net_KVS_delta": round(r.net_kvs_delta, 1),
            "low_confidence_extreme_delta": r.low_confidence_extreme_delta,
            "no_delta_history": r.no_delta_history,
        }
        for r in tier_results
    ])

# Deliberately NOT one table sorted by net_KVS_delta across all six players --
# see the finding directly above. Each reliability_tier gets its own
# independently-ranked table instead.
tiers = group_by_reliability_tier(results)

print(f"=== reliability_tier = '{RELIABILITY_TIER_MODEL}' (real QB/RB/WR/TE predictions, ranked) ===")
print(_to_table(tiers.get(RELIABILITY_TIER_MODEL, [])).to_string(index=False))

print(f"\n=== reliability_tier = '{RELIABILITY_TIER_HEURISTIC}' (K/DEF non-ML heuristic, ranked SEPARATELY -- never against the model tier) ===")
print(_to_table(tiers.get(RELIABILITY_TIER_HEURISTIC, [])).to_string(index=False))


=== reliability_tier = 'model' (real QB/RB/WR/TE predictions, ranked) ===
             player position  predicted_KVS  keeper_cost_VORP  net_KVS_delta  low_confidence_extreme_delta  no_delta_history
         Kyle Pitts       TE           19.8               3.2           16.6                         False             False
     Caleb Williams       QB          -28.6             -18.2          -10.4                         False             False
      Ja'Marr Chase       WR           76.3             107.4          -31.0                         False             False
Christian McCaffrey       RB           74.7             107.4          -32.7                          True             False

=== reliability_tier = 'heuristic' (K/DEF non-ML heuristic, ranked SEPARATELY -- never against the model tier) ===
    player position  predicted_KVS  keeper_cost_VORP  net_KVS_delta  low_confidence_extreme_delta  no_delta_history
   HOU DEF      DEF           45.2             -10.3           55.5  

## Summary

**The draft capital curve looks sensible on real data**: `avg_vorp` declines from +107.4 (round 1) to roughly -15 to -30 through the late rounds, and rounds 16-17 show identical `avg_vorp`/`n_picks` with a shared `band_rounds = (16, 17)` — the banding decision from the investigation, visibly working correctly on the real curve, not just in unit tests.

**The crosswalk resolved 282/310 (91.0%) of fresh 2024-2025 picks on this run** — 100 direct `gsis_id` matches, 186 name/position fallback matches, 19 DEF pass-throughs, and 5 genuine unresolved misses (Joshua Palmer, Marvin Harrison, Kyle Williams, Travis Hunter — name collisions/position mismatches, not chased further).

**All six players produced sensible, honestly-labeled net values**: McCaffrey's negative net value correctly carries `low_confidence_extreme_delta`, tying back to this exact model's documented blind spot for him; Chase's negative value reflects round 1's high bar rather than anything wrong with his prediction; Pitts's positive value is consistent with (and likely conservative relative to) `08d_shap_te.ipynb`'s own real-world caveat about him. The two K/DEF heuristic values are the largest in the table and the least trustworthy — both correctly carry `used_heuristic=True`, and this notebook flags that explicitly rather than letting a sorted-by-value table imply they're the best trade targets.

**No caveat was silently dropped anywhere in this run**: every flagged condition that occurred (McCaffrey's extreme delta, both heuristic predictions) produced a real, printed caveat string alongside the number, confirmed directly in the output above, not assumed from the code.

**The K/DEF ranking finding directly produced a code fix, demonstrated above**: `reliability_tier` is now a first-class, queryable field on every `NetValueResult`, and `group_by_reliability_tier()` ranks the model tier (Pitts, Williams, Chase, McCaffrey) and the heuristic tier (Cam Little, HOU DEF) as two separate lists, never one blended sort — the model tier's best real value here is Kyle Pitts (+16.6), not HOU DEF's +55.5, which now only ever competes against Cam Little's +45.0 within its own, clearly-labeled tier.